# P03 - Preprocesamiento por grupos de jets

Este notebook implementa la variante que separa el espacio de entrada segun la estructura de jets del evento. En lugar de obligar a un unico pipeline global a convivir con variables estructuralmente indefinidas, se crean tres grupos: `G0` para eventos sin jets, `G1` para eventos con un jet y `G2` para eventos con dos o mas jets.

Cada grupo tiene su propio preprocesador, sus propias columnas, medianas, escalador y orden final de features. En `G0` se eliminan variables de leading/subleading jets y de dos jets; en `G1` se eliminan variables de subleading/dijet; en `G2` se elimina `PRI_jet_num` y se conserva la informacion `2 vs 3+` mediante `PRI_jet_num_is_3` sin escalar.

Este notebook no entrena modelos. Deja listos los tres conjuntos procesados para entrenar modelos especializados y recombinar predicciones por `EventId`.

Salidas principales en `data/preprocesor/P03_by_jet_groups/`.



## Imports y configuracion


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_work_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        direct = candidate / "preprocessing_lib.py"
        nested = candidate / "50. preprocesamiento" / "preprocessing_lib.py"
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / "50. preprocesamiento"
    raise FileNotFoundError("No se pudo ubicar 50. preprocesamiento/preprocessing_lib.py")


work_dir = find_work_dir()
if str(work_dir) not in sys.path:
    sys.path.insert(0, str(work_dir))

from preprocessing_lib import (
    build_common_data,
    output_root,
    save_common_artifacts,
    save_global_pipeline_artifacts,
    save_p03_artifacts,
    split_summary,
    transform_global_pipeline,
    transform_p03,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)



## Construccion de grupos y preprocesadores


In [2]:
common = build_common_data()
result = transform_p03(common)
output_dir = output_root() / "P03_by_jet_groups"

print(f"Directorio de salida: {output_dir}")
print("Resumen del split global:")
display(split_summary(common).round(6))

print("Resumen por grupo:")
display(result["group_summary"].round(6))

for group_id, group in result["groups"].items():
    print(f"\n{group_id}: primeras features finales")
    display(pd.DataFrame({"feature": group["metadata"]["final_feature_order"]}).head(25))
    print(f"{group_id}: columnas eliminadas")
    display(pd.DataFrame({"feature": group["metadata"]["drop_columns"]}))


Directorio de salida: /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups
Resumen del split global:


,split,rows,class_b,class_s,weight_sum
0,train,225000,147900.0,77100.0,370822.739836
1,valid,25000,16433.0,8567.0,40869.096094
2,test,550000,NaN,NaN,NaN


Resumen por grupo:


,group,train_rows,valid_rows,test_rows,feature_count,train_nan_count,valid_nan_count,test_nan_count
0,G0,89987,9926,220156,20,0,0,0
1,G1,69731,7813,169716,23,0,0,0
2,G2,65282,7261,160128,31,0,0,0



G0: primeras features finales


,feature
0,DER_mass_MMC
1,DER_mass_transverse_met_lep
2,DER_mass_vis
3,DER_pt_h
4,DER_deltar_tau_lep
5,DER_pt_tot
6,DER_sum_pt
7,DER_pt_ratio_lep_tau
8,DER_met_phi_centrality
9,PRI_tau_pt


... (6 outputs left out)


G2: primeras features finales


,feature
0,DER_mass_MMC
1,DER_mass_transverse_met_lep
2,DER_mass_vis
3,DER_pt_h
4,DER_deltaeta_jet_jet
5,DER_mass_jet_jet
6,DER_prodeta_jet_jet
7,DER_deltar_tau_lep
8,DER_pt_tot
9,DER_sum_pt


G2: columnas eliminadas


,feature
0,PRI_jet_num


## Guardado de artefactos


In [3]:
saved_paths = save_p03_artifacts(common, result, output_dir)
print(f"Artefactos guardados: {len(saved_paths)}")
for path in saved_paths[:30]:
    print(f"- {path}")
if len(saved_paths) > 30:
    print(f"... y {len(saved_paths) - 30} artefactos adicionales")


Artefactos guardados: 48
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups/group_summary.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups/split_summary.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups/G0/X_train_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups/G0/X_valid_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups/G0/X_test_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups/G0/y_train.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups/G0/y_valid.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups/G0/train_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups/G0/valid_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor/P03_by_jet_groups/G0/train_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/preprocesor